# Hyperparameter tuning for the baseline

Finds the client training settings that make the **baseline (synchronous FedAvg)** as strong as possible on the scenario in `settings.yaml` (dataset, non-IID clients, heterogeneous devices). The tuned values are then written back into `settings.yaml`, so every approach is tested against a properly tuned baseline.

**What is tuned:** learning rate, batch size, local epochs, number of clients.

**How every configuration is judged (the fairness rules)**

1. **Same virtual-time budget** (`max_time`). More local epochs or larger batches cost device time, so each configuration gets the same amount of simulated time and spends it differently.
2. **Validation accuracy, not test accuracy.** The score is the mean validation accuracy over the last 10% of the budget, measured on held-out training images that no client holds. The test set is only used once, at the end, to report the chosen configuration.
3. **Same seed for every configuration while searching.** That gives the same data split, devices and starting weights, so configurations are compared like for like. The best candidates are then confirmed on 3 seeds.

> **Note:** a strong baseline means giving the reference method its best settings. If a new approach beats a poorly tuned FedAvg, the comparison proves little.

> **Important:** the number of clients is not an ordinary hyperparameter. It defines the problem (how much data each client has and how many stragglers there are). Stage 3 measures its effect, but choose it for the scenario you want to study, not only for the highest accuracy.

> **Tip:** finished trials are saved to `results/tuning/` and are not re-run. If the notebook is interrupted, run it again and it continues where it stopped.

## 1. Setup

`PARALLEL` trials run at the same time, each with `THREADS` CPU threads. On a 4-core laptop the fastest layout is 4 × 1 for the small MLP and 2 × 2 for the CNNs; it is chosen automatically from the model in `settings.yaml`.

In [ ]:
%run simulator.ipynb
import matplotlib.pyplot as plt
import multiprocessing as mp
plt.style.use("default")

APPROACH = "sync_fedavg"          # the baseline being tuned
TUNING_SEED = 42                  # used for every configuration during the search
CONFIRM_SEEDS = [42, 43, 44]      # used to confirm the best candidates
OUT = RESULTS / "tuning"

BASE = load_settings()
PARALLEL, THREADS = (4, 1) if BASE.model == "simple_nn" else (2, 2)
Baseline = load_approach(APPROACH)
print(f"tuning {APPROACH} on {BASE.dataset}: {BASE.num_clients} clients, budget {BASE.max_time} virtual s, "
      f"{BASE.train_samples} training / {BASE.validation_samples} validation images")

### Trial helper

`trials(configs)` runs each configuration (or loads it if it already exists) and returns one row per configuration with:

* `val_score`: mean validation accuracy over the last 10% of the budget (**the number being maximised**)
* `test_acc`: test accuracy of the final model (reported, never used for choosing)
* `rounds`: how many global updates fit in the budget

In [ ]:
def tag(cfg):
    return "__".join(f"{k}={v}" for k, v in sorted(cfg.items()))


def summarise(cfg, seed, m):
    tail = m[m.time >= 0.9 * BASE.max_time]
    tail = tail if len(tail) else m.tail(3)
    return dict(cfg, seed=seed, val_score=tail.val_accuracy.mean(), test_acc=m.accuracy.iloc[-1],
                rounds=int(m.version.iloc[-1]), wall_s=round(m.wall_seconds.iloc[-1]))


def _one(job):
    cfg, seed = job
    path = OUT / f"{tag(cfg)}_seed{seed}_metrics.csv"
    if path.exists():
        m = pd.read_csv(path)
        if m.settings_id.iloc[0] == settings_id(load_settings(**cfg)):   # same settings -> reuse
            return summarise(cfg, seed, m)
    torch.set_num_threads(THREADS)
    m = run(Baseline(), seeds=[seed], overrides=cfg, results_dir=OUT, name=tag(cfg), verbose=False)
    return summarise(cfg, seed, m)


def trials(configs, seeds=(TUNING_SEED,)):
    jobs = [(cfg, s) for cfg in configs for s in seeds]
    if PARALLEL > 1:
        with mp.get_context("fork").Pool(PARALLEL) as pool:
            rows = pool.map(_one, jobs, chunksize=1)
    else:
        rows = [_one(j) for j in jobs]
    return pd.DataFrame(rows)


def best_of(df):
    return df.sort_values("val_score", ascending=False).iloc[0]

## 2. Stage 1: learning rate and batch size

These two interact strongly (larger batches usually need a larger learning rate), so they are searched together on a grid. Local epochs and clients stay at their current values.

> **Note:** if the best learning rate is at the edge of the grid, the grid is extended by one step in that direction automatically, so the optimum is not cut off.

In [ ]:
LRS = [0.005, 0.01, 0.02, 0.05, 0.1]
BATCH_SIZES = [16, 32, 64]
fixed = dict(local_epochs=BASE.local_epochs, num_clients=BASE.num_clients)

stage1 = trials([dict(fixed, lr=lr, batch_size=bs) for lr in LRS for bs in BATCH_SIZES])
for _ in range(2):                                    # extend the grid if the best lr is on its edge
    b = best_of(stage1)
    lrs = sorted(stage1.lr.unique())
    new_lr = b.lr / 2 if b.lr == lrs[0] else b.lr * 2 if b.lr == lrs[-1] else None
    if new_lr is None:
        break
    print(f"best lr {b.lr} is on the grid edge -> also trying {new_lr}")
    stage1 = pd.concat([stage1, trials([dict(fixed, lr=new_lr, batch_size=bs) for bs in BATCH_SIZES])], ignore_index=True)

best1 = best_of(stage1)
print(f"best: lr={best1.lr}, batch_size={int(best1.batch_size)}  (val {best1.val_score * 100:.2f}%)")
stage1.pivot_table(index="lr", columns="batch_size", values="val_score").mul(100).round(2)

In [ ]:
fig = plt.figure(figsize=(9, 5))
for bs, g in stage1.sort_values("lr").groupby("batch_size"):
    plt.plot(g.lr, g.val_score * 100, marker="o", markersize=3, label=f"batch size {bs}")
plt.xscale("log"); plt.xlabel("Learning rate"); plt.ylabel("Validation accuracy (%)")
plt.title("Stage 1: learning rate and batch size"); plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout()
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "stage1_lr_batch.png", dpi=150, bbox_inches="tight"); plt.show()

## 3. Stage 2: local epochs

More local epochs means more progress per round but fewer rounds in the same time. With non-IID data, too many epochs also make clients drift toward their own data. The learning rate is re-checked around the stage-1 value, because the best learning rate often shifts with the amount of local work.

In [ ]:
EPOCHS = [1, 2, 5]
lr1, bs1 = float(best1.lr), int(best1.batch_size)
cands = [dict(num_clients=BASE.num_clients, batch_size=bs1, local_epochs=e, lr=lr)
         for e in EPOCHS for lr in (lr1 / 2, lr1, lr1 * 2)]
stage2 = trials(cands)
best2 = best_of(stage2)
print(f"best: local_epochs={int(best2.local_epochs)}, lr={best2.lr}  (val {best2.val_score * 100:.2f}%)")
stage2.pivot_table(index="local_epochs", columns="lr", values="val_score").mul(100).round(2)

In [ ]:
fig = plt.figure(figsize=(9, 5))
for lr, g in stage2.sort_values("local_epochs").groupby("lr"):
    plt.plot(g.local_epochs, g.val_score * 100, marker="o", markersize=3, label=f"lr {lr:g}")
plt.xlabel("Local epochs"); plt.ylabel("Validation accuracy (%)"); plt.xticks(EPOCHS)
plt.title(f"Stage 2: local epochs (batch size {bs1})"); plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout()
fig.savefig(OUT / "stage2_epochs.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Stage 3: number of clients

The same 20,000 images are spread over more or fewer clients. More clients means less data each, more non-IID skew per client, and a higher chance that some client is very slow (the synchronous round waits for it).

> **Important:** read this as a sensitivity study. The tuner picks the value with the best validation score, but you can override it below (`CHOSEN_CLIENTS`) if you want a scenario with more clients for straggler research.

In [ ]:
CLIENTS = [10, 20, 50]
e2, lr2 = int(best2.local_epochs), float(best2.lr)
stage3 = trials([dict(num_clients=n, batch_size=bs1, local_epochs=e2, lr=lr2) for n in CLIENTS])
best3 = best_of(stage3)
stage3[["num_clients", "val_score", "test_acc", "rounds"]].assign(val_score=lambda d: (d.val_score * 100).round(2))

In [ ]:
fig = plt.figure(figsize=(9, 5))
g = stage3.sort_values("num_clients")
plt.plot(g.num_clients, g.val_score * 100, marker="o", markersize=3, label="validation accuracy")
plt.xlabel("Number of clients"); plt.ylabel("Validation accuracy (%)"); plt.xticks(CLIENTS)
plt.title("Stage 3: number of clients"); plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout()
fig.savefig(OUT / "stage3_clients.png", dpi=150, bbox_inches="tight"); plt.show()

CHOSEN_CLIENTS = int(best3.num_clients)     # override here if you want a different scenario size
print("number of clients:", CHOSEN_CLIENTS)

## 5. Confirm on 3 seeds

The single-seed search can be fooled by luck. The three best distinct configurations found so far are re-run on 3 seeds, and the final choice is the best **mean** validation score.

> **Note:** each seed changes the data split, the device speeds and the starting weights. A configuration that wins on average over seeds is a real improvement, not a lucky draw.

In [ ]:
keys = ["num_clients", "batch_size", "local_epochs", "lr"]
pool = pd.concat([stage1, stage2, stage3], ignore_index=True)
pool = pool[pool.num_clients == CHOSEN_CLIENTS] if (pool.num_clients == CHOSEN_CLIENTS).any() else pool
top = pool.sort_values("val_score", ascending=False).drop_duplicates(keys).head(3)
top_cfgs = [dict(num_clients=int(r.num_clients), batch_size=int(r.batch_size), local_epochs=int(r.local_epochs), lr=float(r.lr))
            for r in top.itertuples()]
confirm = trials(top_cfgs, seeds=CONFIRM_SEEDS)
table = (confirm.groupby(keys).agg(val_mean=("val_score", "mean"), val_std=("val_score", "std"),
                                   test_mean=("test_acc", "mean"), test_std=("test_acc", "std"), rounds=("rounds", "mean"))
         .sort_values("val_mean", ascending=False))
display((table * [100, 100, 100, 100, 1]).round(2))
chosen = dict(zip(keys, table.index[0]))
chosen = {k: (float(v) if k == "lr" else int(v)) for k, v in chosen.items()}
print("chosen configuration:", chosen)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cfg in top_cfgs:
    curves = [pd.read_csv(OUT / f"{tag(cfg)}_seed{s}_metrics.csv") for s in CONFIRM_SEEDS]
    c = pd.concat(curves).groupby("version", as_index=False)[["time", "val_accuracy", "accuracy"]].mean()
    label = ", ".join(f"{k}={v}" for k, v in cfg.items())
    axes[0].plot(c.time, c.val_accuracy * 100, marker="o", markersize=3, label=label)
    axes[1].plot(c.time, c.accuracy * 100, marker="o", markersize=3, label=label)
axes[0].set(xlabel="Elapsed virtual time (s)", ylabel="Validation accuracy (%)", title="Validation accuracy vs simulated time")
axes[1].set(xlabel="Elapsed virtual time (s)", ylabel="Test accuracy (%)", title="Test accuracy vs simulated time")
for ax in axes:
    ax.grid(alpha=.3)
axes[1].legend(loc="best", fontsize=8)
plt.tight_layout(); fig.savefig(OUT / "confirmation_curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Save the tuned values into `settings.yaml`

Only the four tuned values are changed; comments and every other setting stay as they are.

> **Caution:** this changes the shared settings, so results made before this point no longer match and `compare.ipynb` will skip them. Re-run the approaches (`run_all.ipynb`) afterwards.

In [ ]:
import re
WRITE_SETTINGS = True

text = (ROOT / "settings.yaml").read_text()
for key, value in chosen.items():
    text = re.sub(rf"^({key}:\s*)\S+", rf"\g<1>{value}", text, count=1, flags=re.M)
if WRITE_SETTINGS:
    (ROOT / "settings.yaml").write_text(text)
    print("settings.yaml updated:", chosen)
else:
    print("not written (WRITE_SETTINGS = False). Values would be:", chosen)